# 🎬 Intelligent Video Editing System

**Autonomous AI video editor** – runs entirely in Google Colab, no installation needed.

### What this notebook does
1. Installs all dependencies automatically
2. Clones the project code
3. Mounts your Google Drive for persistent storage
4. Launches an interactive Gradio web UI

> 💡 **Tip:** Use **Runtime → Change runtime type → GPU** for faster processing.

---

In [ ]:
# ============================================================
# Cell 1 – Install dependencies
# ============================================================
# Run this cell first. It takes ~2 minutes on a fresh Colab runtime.

import subprocess, sys

print('Installing system packages (ffmpeg) …')
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)

print('Installing Python packages …')
packages = [
    'opencv-python-headless>=4.8.0',
    'moviepy>=1.0.3',
    'imageio>=2.31.0',
    'imageio-ffmpeg>=0.4.9',
    'librosa>=0.10.0',
    'soundfile>=0.12.1',
    'numpy>=1.24.0',
    'gradio>=4.0.0',
    'tqdm>=4.66.0',
    'Pillow>=10.0.0',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages,
    check=True
)

print('\n✅ All dependencies installed successfully!')

In [ ]:
# ============================================================
# Cell 2 – Clone the project (skip if already cloned)
# ============================================================
import os

REPO_DIR = '/content/intelligent-video-editing-system'

if not os.path.isdir(REPO_DIR):
    print('Cloning repository …')
    subprocess.run([
        'git', 'clone', '-q',
        'https://github.com/Karthickkavin/intelligent-video-editing-system.git',
        REPO_DIR
    ], check=True)
    print('Repository cloned.')
else:
    print('Repository already present; pulling latest …')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)

# Add to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ============================================================
# Cell 3 – Mount Google Drive (optional but recommended)
# ============================================================
# Mounting Drive lets you:
#   • Upload videos from Drive instead of re-uploading every session
#   • Save edited videos directly to Drive
#
# Skip this cell if you prefer to upload files directly.

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = '/content/drive/MyDrive/VideoEditor'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'✅ Drive mounted. Your VideoEditor folder: {DRIVE_ROOT}')
except Exception as e:
    print(f'⚠️  Drive mount skipped ({e}). Files will be saved locally in /content/.')

In [ ]:
# ============================================================
# Cell 4 – GPU check and system info
# ============================================================
import platform

print('Python  :', sys.version.split()[0])
print('Platform:', platform.platform())

try:
    import torch  # type: ignore
    cuda_ok = torch.cuda.is_available()
    print('PyTorch :', torch.__version__)
    print('CUDA    :', 'available ✅' if cuda_ok else 'not available (CPU mode)')
    if cuda_ok:
        print('GPU     :', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch : not installed (optional)')

# Check FFmpeg
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
ffmpeg_line = result.stdout.splitlines()[0] if result.returncode == 0 else 'NOT FOUND'
print('FFmpeg  :', ffmpeg_line)

In [ ]:
# ============================================================
# Cell 5 – Launch the Gradio Web UI
# ============================================================
# After running this cell a public URL will appear below.
# Open it to access the video editor interface.

import gradio as gr  # type: ignore
import tempfile
import shutil

# ----- Import our pipeline -----
from utils.logger import setup_logger
from example_usage import run_pipeline

setup_logger()  # initialise logging (writes to logs/video_editor.log)

# ---- Process function called by Gradio ----
def process_video(
    video_file,
    quality,
    intensity,
    enable_scene_detection,
    enable_silence_removal,
    enable_transitions,
    enable_speed_adjustment,
    enable_auto_zoom,
    enable_color_correction,
    enable_audio_normalization,
    progress=gr.Progress(track_tqdm=True),
):
    """Main Gradio callback."""
    if video_file is None:
        raise gr.Error('Please upload a video file first.')

    input_path = video_file
    output_path = tempfile.mktemp(suffix='_edited.mp4', dir='/tmp')

    progress(0, desc='Starting …')

    def _cb(pct, msg):
        progress(pct / 100, desc=msg)

    try:
        from utils.config import Config
        from ai_engine import Analyzer, VideoEditor, Renderer

        cfg = Config(
            quality=quality,
            intensity=intensity,
            enable_scene_detection=enable_scene_detection,
            enable_silence_removal=enable_silence_removal,
            enable_transitions=enable_transitions,
            enable_speed_adjustment=enable_speed_adjustment,
            enable_auto_zoom=enable_auto_zoom,
            enable_color_correction=enable_color_correction,
            enable_audio_normalization=enable_audio_normalization,
        )

        _cb(5, 'Analysing video …')
        analysis = Analyzer(cfg).analyze(input_path)

        _cb(35, 'Creating edit plan …')
        edit_plan = VideoEditor(cfg).create_edit_plan(analysis)

        _cb(45, 'Rendering …')

        def _render_cb(pct, msg):
            _cb(45 + int(pct * 0.55), msg)

        Renderer(cfg).render(input_path, edit_plan, output_path, progress_callback=_render_cb)

        _cb(100, 'Done!')

        # Stats
        scenes = len(analysis.get('scenes', []))
        silences = sum(1 for s in analysis.get('audio_segments', []) if s.get('is_silence'))
        effects = len(edit_plan.get('effects', []))
        stats = (
            f'✅ Editing complete!\n'
            f'Scenes detected : {scenes}\n'
            f'Silences removed: {silences}\n'
            f'Effects applied : {effects}\n'
            f'Output quality  : {quality}'
        )

        # Save to Drive if available
        try:
            drive_out = f'/content/drive/MyDrive/VideoEditor/edited_{os.path.basename(output_path)}'
            if os.path.isdir('/content/drive/MyDrive/VideoEditor'):
                shutil.copy2(output_path, drive_out)
                stats += f'\nSaved to Drive : {drive_out}'
        except Exception:
            pass

        return output_path, stats

    except Exception as exc:
        raise gr.Error(f'Processing failed: {exc}') from exc


# ---- Build the Gradio interface ----
with gr.Blocks(
    title='Intelligent Video Editing System',
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown(
        '# 🎬 Intelligent Video Editing System\n'
        '### Autonomous AI video editor – upload your video and let AI do the editing!'
    )

    with gr.Row():
        # Left column – inputs
        with gr.Column(scale=1):
            gr.Markdown('## 📤 Input')
            video_input = gr.Video(
                label='Upload Video (MP4, MOV, AVI)',
                sources=['upload'],
            )

            gr.Markdown('## ⚙️ Settings')
            quality_dd = gr.Dropdown(
                choices=['480p', '720p', '1080p'],
                value='720p',
                label='Output Quality',
            )
            intensity_dd = gr.Dropdown(
                choices=['light', 'medium', 'heavy'],
                value='medium',
                label='Editing Intensity',
            )

            gr.Markdown('## 🔧 Features')
            cb_scene  = gr.Checkbox(value=True,  label='Scene Detection')
            cb_silence = gr.Checkbox(value=True, label='Silence Removal')
            cb_trans  = gr.Checkbox(value=True,  label='Smart Transitions')
            cb_speed  = gr.Checkbox(value=True,  label='Speed Adjustment')
            cb_zoom   = gr.Checkbox(value=True,  label='Auto-Zoom on Faces')
            cb_color  = gr.Checkbox(value=True,  label='Colour Correction')
            cb_audio  = gr.Checkbox(value=True,  label='Audio Normalisation')

            edit_btn = gr.Button('🎬 Edit Video', variant='primary', size='lg')

        # Right column – outputs
        with gr.Column(scale=1):
            gr.Markdown('## 📥 Output')
            video_output = gr.Video(label='Edited Video')
            stats_output = gr.Textbox(
                label='Processing Summary',
                lines=8,
                interactive=False,
            )

    gr.Markdown(
        '---\n'
        '**Tips:**  '
        'Use **GPU runtime** (Runtime → Change runtime type → T4 GPU) for faster processing.  '
        'Large files (>500 MB) may cause memory issues on the free tier – use 480p quality.'
    )

    edit_btn.click(
        fn=process_video,
        inputs=[
            video_input, quality_dd, intensity_dd,
            cb_scene, cb_silence, cb_trans,
            cb_speed, cb_zoom, cb_color, cb_audio,
        ],
        outputs=[video_output, stats_output],
    )

print('Launching Gradio …')
demo.launch(share=True, debug=False)

In [ ]:
# ============================================================
# Cell 6 – Command-line usage (alternative to the UI)
# ============================================================
# Edit a video programmatically without the Gradio UI.

# Replace with the path to your video (e.g. from Drive)
INPUT_VIDEO  = '/content/my_video.mp4'
OUTPUT_VIDEO = '/content/my_video_edited.mp4'

# Uncomment and run after uploading a video
# from example_usage import run_pipeline
# result = run_pipeline(
#     input_path=INPUT_VIDEO,
#     output_path=OUTPUT_VIDEO,
#     quality='720p',
#     intensity='medium',
#     progress_callback=lambda pct, msg: print(f'[{pct:3.0f}%] {msg}'),
# )
# print(f'Output: {result}')

In [ ]:
# ============================================================
# Cell 7 – Batch processing multiple videos
# ============================================================

# INPUT_VIDEOS = [
#     '/content/drive/MyDrive/clips/clip1.mp4',
#     '/content/drive/MyDrive/clips/clip2.mp4',
# ]
# OUTPUT_DIR = '/content/drive/MyDrive/VideoEditor/batch_output'

# from example_usage import run_batch
# results = run_batch(INPUT_VIDEOS, output_dir=OUTPUT_DIR, quality='720p', intensity='medium')
# print('Batch complete:', results)